# AnimationStudio - ComfyUI generation on Google Colab (T4)

This notebook installs the studio plus ComfyUI on a Colab GPU runtime
(Tesla T4, 16 GB VRAM), downloads the model for the selected branch,
starts ComfyUI, runs Phase-1 generation, exports real PNGs into your
Google Drive, and launches the Review UI.

## Branch = model flavor

| Branch | Model | Notes |
| --- | --- | --- |
| `colab-gpu` | fp8 Flux dev bundle (`flux1-dev.safetensors`, ~12 GB) | Best quality on the T4. One-file `CheckpointLoaderSimple`. |
| `master` | Q4 GGUF (`flux1-dev-Q4_K_S.gguf` + encoders/VAE, ~14 GB) | CPU-grade; also runs on the T4. Needs `ComfyUI-GGUF`. |

## Steps

1. Runtime -> Change runtime type -> T4 GPU (or better).
2. In Cell 1 set `REPO_URL` to your GitHub clone URL.
3. Runtime -> Run all. First run downloads the model into your Drive so later sessions are fast.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (push master + colab-gpu there first).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# colab-gpu -> fp8 Flux (16GB VRAM, best on T4).  master -> Q4 GGUF (CPU-grade).
BRANCH = "colab-gpu"  #@param ["colab-gpu", "master"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
UI_PORT = 8000  #@param {type:"integer"}

# Models, catalog.db and exported images persist here across sessions.
DRIVE_ROOT = "/content/drive/MyDrive/AnimationStudio"  #@param {type:"string"}

# Generation scope.  Keep small on the free tier (~2 min/image on a T4).
GENERATION_ARGS = '--characters "Lily Bunny" --asset-types expressions poses --count 2 --shortlist 1 --fast-scoring'  #@param {type:"string"}
EXPORT_SCOPE = "characters"  #@param ["characters", "environments", "vehicles", "backgrounds", "props", "all"]
EXPORT_TYPES = ""  #@param {type:"string"}
EXPORT_SIZE = 1024  #@param {type:"integer"}

START_TUNNEL = True  #@param {type:"boolean"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"
DB = f"{DRIVE_ROOT}/catalog.db"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 to your GitHub repository before running.")


In [ ]:
#@title 2. Mount Google Drive

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
#@title 3. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])

# Studio first (torch is already preinstalled on Colab), then light deps.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn", "timm", "diffusers", "transformers"])
print("Studio installed (branch:", BRANCH, ")")


In [ ]:
#@title 4. Install ComfyUI (and ComfyUI-GGUF on the GGUF branch)

if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])

if BRANCH == "master":
    gguf = f"{COMFY}/custom_nodes/ComfyUI-GGUF"
    if not os.path.isdir(gguf):
        run(["git", "clone", "--depth", "1",
             "https://github.com/city96/ComfyUI-GGUF.git", gguf])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{gguf}/requirements.txt"])
print("ComfyUI ready at", COMFY)


In [ ]:
#@title 5. Download models (cached in Drive, linked into ComfyUI)

MODELS = {
    "colab-gpu": {
        "checkpoints/flux1-dev.safetensors":
            "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    },
    "master": {
        "checkpoints/flux1-dev-Q4_K_S.gguf":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q4_K_S.gguf",
        "clip/clip_l.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/clip_l.safetensors",
        "clip/t5xxl_fp16.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/t5xxl_fp16.safetensors",
        "vae/ae.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/ae.safetensors",
    },
}[BRANCH]

import shutil

for rel, url in MODELS.items():
    cached = f"{DRIVE_ROOT}/models/{rel}"
    link = f"{COMFY}/models/{rel}"
    if not (os.path.exists(cached) and os.path.getsize(cached) > 0):
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        print(f"Downloading {rel} ...")
        run(["wget", "-q", "-c", "-O", cached, url])
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.lexists(link) and not os.path.islink(link):
        os.remove(link)
    if not os.path.islink(link):
        try:
            os.symlink(cached, link)
        except OSError:
            shutil.copyfile(cached, link)
    print(f"OK {rel} ({os.path.getsize(cached) / 1e9:.2f} GB)")


In [ ]:
#@title 6. Start the ComfyUI server

import time
import urllib.request


def server_up(port):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=3)
        return True
    except Exception:
        return False


if server_up(COMFYUI_PORT):
    print("ComfyUI already running on", COMFYUI_PORT)
else:
    logf = open(f"{WORK}/comfyui.log", "a")
    server = subprocess.Popen(
        [sys.executable, "main.py", "--port", str(COMFYUI_PORT), "--listen", "127.0.0.1"],
        cwd=COMFY, stdout=logf, stderr=subprocess.STDOUT,
    )
    for _ in range(120):
        if server_up(COMFYUI_PORT):
            break
        time.sleep(2)
    up = server_up(COMFYUI_PORT)
    print("ComfyUI", "UP" if up else "FAILED", f"-> http://localhost:{COMFYUI_PORT}")
    if not up:
        print("---- tail comfyui.log ----")
        print(open(f"{WORK}/comfyui.log").read()[-3000:])


In [ ]:
#@title 7. Verify the GPU

import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
#@title 8. Run Phase-1 generation (ComfyUI backend)

os.chdir(REPO)
!python scripts/generate_phase1_library.py --backend comfyui --comfyui-url http://localhost:{COMFYUI_PORT} --db {DB} {GENERATION_ARGS}


In [ ]:
#@title 9. Export real PNGs into the Drive file tree

os.chdir(REPO)
etype = f"--asset-types {EXPORT_TYPES}" if EXPORT_TYPES else ""
!python scripts/export_assets.py --db {DB} --backend comfyui --comfyui-url http://localhost:{COMFYUI_PORT} --scope {EXPORT_SCOPE} {etype} --size {EXPORT_SIZE} --universe {DRIVE_ROOT}/Universe --world {DRIVE_ROOT}/World --assets {DRIVE_ROOT}/Assets


In [ ]:
#@title 10. Launch the Review UI and tunnel

from src.review_ui.app import create_app
from src.universe.batch_generator import resolve_backend

app = create_app(
    db_path=DB,
    generation_backend=resolve_backend("comfyui", comfyui_url=f"http://localhost:{COMFYUI_PORT}"),
    universe_dir=f"{DRIVE_ROOT}/Universe",
    world_dir=f"{DRIVE_ROOT}/World",
    assets_dir=f"{DRIVE_ROOT}/Assets",
)

import threading
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=UI_PORT, log_level="warning")
threading.Thread(target=uvicorn.Server(config).run, daemon=True).start()
print(f"Review UI starting on :{UI_PORT} ...")

if START_TUNNEL:
    def open_tunnel(port, name):
        out = open(f"{WORK}/{name}.log", "w")
        return subprocess.Popen(
            ["npx", "--yes", "localtunnel", "--port", str(port)],
            stdout=out, stderr=subprocess.STDOUT,
        )

    open_tunnel(COMFYUI_PORT, "tunnel_comfyui")
    open_tunnel(UI_PORT, "tunnel_ui")
    time.sleep(8)
    for name in ("tunnel_comfyui", "tunnel_ui"):
        lines = open(f"{WORK}/{name}.log").read().splitlines()
        urls = [ln.strip() for ln in lines if "loca.lt" in ln]
        print(name, "->", urls)


## Next steps

- All state lives under `DRIVE_ROOT` (`catalog.db`, `models/`, `Universe/`, `World/`, `Assets/`) and survives session resets.
- Extend the library: change `GENERATION_ARGS` in Cell 1, re-run Cells 8-10. Already-shortlisted variants are skipped (idempotent).
- Browse the Review UI tunnel URL to approve/lock/shortlist assets.
- The ComfyUI tunnel URL opens the raw ComfyUI web UI.

## Troubleshooting

- CUDA out of memory: shrink the scope in Cell 1, or switch to the L4/A100 runtime; alternatively set `BRANCH = "master"` (GGUF).
- Model download stalls: re-run Cell 5 (`wget -c` resumes into the Drive cache).
- ComfyUI failed to start: read the log tail printed by Cell 6.
- Runtime disconnect: restart and run Cells 2-10; models are cached on Drive.
